# CS383: Data Science and Machine Learning
## Lecture 3 Exercises — Pandas and SQL Fundamentals

**Make a copy of this notebook before you start** (do not edit this original). Fill in every `__________` blank, then run all cells top to bottom before submitting. This notebook is graded with Otter Grader — do not delete or modify the setup cells.

### Setup

Run this first — it rebuilds the same NYC 311 dataset used in Lecture 3, as both a Pandas DataFrame and a SQLite table.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import requests

SOCRATA_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

try:
    response = requests.get(
        SOCRATA_URL,
        params={
            "$limit": 8000,
            "$order": "created_date DESC",
            "$select": "unique_key,complaint_type,borough,created_date",
        },
        timeout=8,
    )
    response.raise_for_status()
    complaints_df = pd.DataFrame(response.json())
    live = True
except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    complaints_df = pd.DataFrame({
        "unique_key": np.arange(1, n + 1),
        "complaint_type": rng.choice(
            complaint_types, size=n,
            p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
        ),
        "borough": rng.choice(boroughs, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": pd.date_range("2026-01-01", periods=n, freq="min").astype(str),
    })
    live = False

print(f"{'Live' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")

# Write the exact same data into a real SQLite database file.
conn = sqlite3.connect("nyc311_lab.db")
complaints_df.to_sql("complaints", conn, if_exists="replace", index=False)

print("Wrote table 'complaints' into nyc311_lab.db")

---

## Exercise 1 — Ask Your Own Question, Two Ways

**Scenario:** you want to know the top 5 complaint types in Queens specifically.

Fill in the blanks below to answer this first in SQL, then in Pandas, then check that both approaches agree.

### Step 1 — SQL

In [ ]:
sql = """
SELECT complaint_type, COUNT(*) AS n
FROM complaints
WHERE borough = 'QUEENS'
GROUP BY __________
ORDER BY __________ DESC
LIMIT 5;
"""
your_sql_result = pd.read_sql_query(sql, conn)
print(your_sql_result)

### Step 2 — Pandas

In [ ]:
queens_only = complaints_df[complaints_df["borough"] == "__________"]
your_pandas_result = queens_only["complaint_type"].value_counts().head(5)
print(your_pandas_result)

### Step 3 — Do they match?

In [ ]:
match = list(your_sql_result["complaint_type"]) == list(your_pandas_result.index)
print(f"Results match: {match}")

---

## Exercise 2 — Reflection (Exit Ticket)

Answer the following in your own words.

1. How does a SQL table relate to a Pandas DataFrame?
2. In plain English, what would `WHERE borough = 'BRONX' AND complaint_type = 'Rodent'` return?
3. Why can't you use Python's `and`/`or` inside a Pandas boolean filter?
4. What's one question about this dataset you'd want to answer with a `GROUP BY`?
5. What question do you still have about SQL or Pandas before Lecture 4 (joins, datetime, real-world cleaning)?

**Your responses:**

1.  
2.  
3.  
4.  
5.  

## Optional Challenge

Pick a borough and complaint type combination you're curious about.

### Step 1 — Choose your borough and complaint type

In [ ]:
my_borough = "__________"          # e.g. "BRONX"
my_complaint_type = "__________"   # e.g. "Illegal Parking"


### Step 2 — Answer your question in SQL

In [ ]:
# Write a SQL query using my_borough and my_complaint_type
# Your code here


### Step 3 — Answer the same question in Pandas

In [ ]:
# Your code here


### Step 4 — Confirm the two results match

In [ ]:
# Your code here


### Step 5 — Swap AND for OR

Modify your `WHERE` condition to use `OR` instead of `AND` (and its Pandas equivalent, `|` instead of `&`). In one sentence, explain how and why the results changed.

In [ ]:
# Your code here


**Your explanation:**

